# 2 · Validate the router (a real regression check, not a demo)

Before trusting `config.PROJECT_DESCRIPTIONS` or `config.SIMILARITY_THRESHOLD` in
production, this runs every question in `data/eval_set_starter.csv` — real questions
with a known correct project — through the router and checks whether it actually picks
the right one. If you've edited a description or the threshold, **this is what catches
a routing regression before it reaches the live services**, not after.

Doesn't need the index from notebook 1 — the router only compares question embeddings
against `PROJECT_DESCRIPTIONS`, so this can run on its own.

In [ ]:
%%capture
!pip install -q -r requirements.txt


In [ ]:
# Get the project files (config.py, portfolio.py, data/) if they aren't
# already here -- lets this notebook be opened and run on its own in Colab.
import os, subprocess, sys

if not os.path.exists("portfolio.py"):
    if os.path.exists("../portfolio.py"):
        os.chdir("..")
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/hossamhamdy333/AI_Portfolio.git", "repo"],
            check=True,
        )
        os.chdir("repo/Codebase_Insight_Agent")

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())


In [ ]:
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google API key (Gemini): ")

# QDRANT_URL/QDRANT_API_KEY make the index PERSISTENT (Qdrant Cloud, free
# tier is enough) instead of rebuilt from scratch every time. This matters
# specifically because this notebook runs in a fresh Colab VM each time,
# separate from wherever mcp_server.py/web_app.py actually run - without a
# real, shared QDRANT_URL, this notebook's work never reaches those
# services at all. Get a free instance at https://cloud.qdrant.io
if not os.environ.get("QDRANT_URL"):
    os.environ["QDRANT_URL"] = input("Qdrant Cloud URL (blank = local in-memory, no persistence): ")
if os.environ["QDRANT_URL"] and not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass("Qdrant API key: ")


In [ ]:
import config
import portfolio

router = portfolio.build_router()


In [ ]:
import pandas as pd

eval_questions = pd.read_csv("data/eval_set_starter.csv")
print(f"{len(eval_questions)} regression questions loaded")
eval_questions[["question", "expected_projects"]]


Run every regression question through the router and check it against the known-correct project:

In [ ]:
results = []
for _, row in eval_questions.iterrows():
    expected = [p.strip() for p in row["expected_projects"].split(",")]
    picked = router.select(row["question"])
    # correct means the router found EVERY expected project, not just one -
    # a few rows expect two projects at once (a question comparing
    # rag_router and rag-vanilla-vs-langchain), and missing either one is
    # a real miss, not a partial pass.
    correct = all(p in picked for p in expected)
    results.append({"question": row["question"][:60], "expected": expected, "picked": picked, "correct": correct})

results_df = pd.DataFrame(results)
pass_count = results_df["correct"].sum()
print(f"{pass_count}/{len(results_df)} routed correctly\n")
results_df


**If anything shows `correct=False` above**, that's a real regression — either a
`PROJECT_DESCRIPTIONS` entry needs to be more specific, or `SIMILARITY_THRESHOLD` needs
adjusting. Don't move on to notebook 3 with failures here; the agent can't route a
question correctly if the router underneath it can't.

In [ ]:
failures = results_df[~results_df["correct"]]
if len(failures) > 0:
    print(f"{len(failures)} routing failure(s):")
    display(failures)
else:
    print("All regression questions routed correctly.")
